# Day 10 v1 — Gold: Sessions Enriched (API Silver → Gold)

**Learning version — one Gold mart, every join step explicit.**

**Source (Silver API):**
- `silver-volume/api/sessions/`
- `silver-volume/api/customers/`
- `silver-volume/api/vehicles/`
- `silver-volume/api/chargers/`
- `silver-volume/api/stations/`
- `silver-volume/api/tariffs/`
- `silver-volume/api/payments/`

**Sink:** `gold-volume/api/sessions_enriched/` (Delta, full overwrite)

**What this mart answers:**
- One row per charging session with full customer, vehicle, charger, station, tariff and payment context
- Enables: revenue per session, energy analysis, customer behaviour, charger utilisation

**Steps:**
1. Read all Silver tables
2. Left-join sessions → customers, vehicles, chargers, stations, tariffs, payments
3. Compute derived metrics (cost_aud, cost_per_kwh, utilisation_pct)
4. Add Gold audit columns
5. Write Gold Delta (full overwrite)
6. Verify output

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
SILVER_API  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD_API    = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/api'
PIPELINE    = 'pl_gold_api_sessions_enriched_v1'
RUN_TS      = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Silver : {SILVER_API}')
print(f'Gold   : {GOLD_API}')
print(f'RUN_TS : {RUN_TS}')

In [ ]:
# Cell 3 — Read Silver tables
sessions  = spark.read.format('delta').load(f'{SILVER_API}/sessions')
customers = spark.read.format('delta').load(f'{SILVER_API}/customers')
vehicles  = spark.read.format('delta').load(f'{SILVER_API}/vehicles')
chargers  = spark.read.format('delta').load(f'{SILVER_API}/chargers')
stations  = spark.read.format('delta').load(f'{SILVER_API}/stations')
tariffs   = spark.read.format('delta').load(f'{SILVER_API}/tariffs')
payments  = spark.read.format('delta').load(f'{SILVER_API}/payments')

print(f'sessions  : {sessions.count():>8} rows')
print(f'customers : {customers.count():>8} rows')
print(f'vehicles  : {vehicles.count():>8} rows')
print(f'chargers  : {chargers.count():>8} rows')
print(f'stations  : {stations.count():>8} rows')
print(f'tariffs   : {tariffs.count():>8} rows')
print(f'payments  : {payments.count():>8} rows')

In [ ]:
# Cell 4 — Prepare dimension tables (select only columns needed for join)

# customers: keep identity + location
cust_dim = customers.select(
    F.col('customer_id'),
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('email').alias('customer_email'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
    F.col('country').alias('customer_country'),
)

# vehicles: keep vehicle specs
veh_dim = vehicles.select(
    F.col('vehicle_id'),
    F.col('make').alias('vehicle_make'),
    F.col('model').alias('vehicle_model'),
    F.col('year').alias('vehicle_year'),
    F.col('battery_capacity').alias('battery_capacity_kwh'),
    F.col('range_km'),
)

# chargers: type + power
chg_dim = chargers.select(
    F.col('charger_id'),
    F.col('charger_type'),
    F.col('power_kw').alias('charger_power_kw'),
    F.col('connector_type'),
    F.col('status').alias('charger_status'),
)

# stations: location
sta_dim = stations.select(
    F.col('station_id'),
    F.col('name').alias('station_name'),
    F.col('city').alias('station_city'),
    F.col('state').alias('station_state'),
    F.col('country').alias('station_country'),
    F.col('latitude').alias('station_lat'),
    F.col('longitude').alias('station_lon'),
)

# tariffs: pricing
tar_dim = tariffs.select(
    F.col('tariff_id'),
    F.col('name').alias('tariff_name'),
    F.col('price_per_kwh').alias('tariff_price_per_kwh'),
    F.col('price_per_min').alias('tariff_price_per_min'),
    F.col('currency').alias('tariff_currency'),
)

# payments: one payment per session (latest by created_at)
from pyspark.sql.window import Window
pay_window = Window.partitionBy('session_id').orderBy(F.col('created_at').desc())
pay_dim = (
    payments
    .withColumn('_rn', F.row_number().over(pay_window))
    .filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('session_id'),
        F.col('payment_id'),
        F.col('amount_aud'),
        F.col('status').alias('payment_status'),
        F.col('payment_method'),
    )
)

print('Dimension tables prepared')

In [ ]:
# Cell 5 — Join sessions with all dimensions
enriched = (
    sessions
    .join(cust_dim, on='customer_id', how='left')
    .join(veh_dim,  on='vehicle_id',  how='left')
    .join(chg_dim,  on='charger_id',  how='left')
    .join(sta_dim,  on='station_id',  how='left')
    .join(tar_dim,  on='tariff_id',   how='left')
    .join(pay_dim,  on='session_id',  how='left')
)

print(f'Enriched rows : {enriched.count()}')
print(f'Columns       : {len(enriched.columns)}')

In [ ]:
# Cell 6 — Compute derived metrics
gold_df = (
    enriched
    # Session date parts for easy partitioning / filtering
    .withColumn('session_date',  F.to_date(F.col('start_time')))
    .withColumn('session_year',  F.year(F.col('start_time')))
    .withColumn('session_month', F.month(F.col('start_time')))
    .withColumn('session_hour',  F.hour(F.col('start_time')))
    # Cost derived: prefer actual payment amount, fall back to tariff × energy
    .withColumn(
        'cost_aud',
        F.coalesce(
            F.col('amount_aud'),
            (F.col('tariff_price_per_kwh') * F.col('energy_kwh')
             + F.col('tariff_price_per_min') * F.col('duration_minutes')).cast('decimal(10,2)')
        )
    )
    # Cost per kWh (avoid divide-by-zero)
    .withColumn(
        'cost_per_kwh',
        F.when(
            F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0),
            (F.col('cost_aud') / F.col('energy_kwh')).cast('decimal(8,4)')
        )
    )
    # Battery fill pct (energy delivered as % of battery capacity)
    .withColumn(
        'battery_fill_pct',
        F.when(
            F.col('battery_capacity_kwh').isNotNull() & (F.col('battery_capacity_kwh') > 0),
            (F.col('energy_kwh') / F.col('battery_capacity_kwh') * 100).cast('decimal(6,2)')
        )
    )
    # Audit
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'Gold rows  : {gold_df.count()}')
print(f'Gold cols  : {len(gold_df.columns)}')
gold_df.printSchema()

In [ ]:
# Cell 7 — Write Gold Delta (full overwrite)
gold_path = f'{GOLD_API}/sessions_enriched'

(
    gold_df.write.format('delta')
    .mode('overwrite').option('overwriteSchema', 'true')
    .partitionBy('session_year', 'session_month')
    .save(gold_path)
)

print(f'Written to : {gold_path}')
print(f'Rows       : {gold_df.count()}')

In [ ]:
# Cell 8 — Verify Gold output
g = spark.read.format('delta').load(gold_path)
print(f'Total Gold rows : {g.count()}')
g.show(3, truncate=True)

print('\nRevenue by station (top 5):')
g.groupBy('station_name').agg(
    F.count('session_id').alias('sessions'),
    F.round(F.sum('cost_aud'), 2).alias('total_revenue_aud'),
    F.round(F.sum('energy_kwh'), 2).alias('total_energy_kwh'),
).orderBy(F.col('total_revenue_aud').desc()).show(5, truncate=False)

print('\nNull check on session_id (should be 0):')
print(g.filter(F.col('session_id').isNull()).count())